# 05 - Evaluacion del Error: R^2 y Error Cuadratico Medio (MSE)

---

## Contenido

1. Por que evaluar un modelo
2. Coeficiente de Determinacion ($R^2$)
3. Error Cuadratico Medio (MSE) y RMSE
4. Otras metricas complementarias
5. Division entrenamiento/prueba y validacion
6. Ejemplo completo con el dataset Iris

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set()

## 1. Por que evaluar un modelo

Una vez que hemos ajustado un modelo de regresion, necesitamos responder una pregunta fundamental: **que tan bien funciona?**

La evaluacion del modelo nos permite:

- Cuantificar la calidad de las predicciones.
- Comparar diferentes modelos de manera objetiva.
- Detectar problemas como sobreajuste o subajuste.
- Tomar decisiones informadas sobre cual modelo usar en produccion.

Las dos metricas mas utilizadas en regresion son el **Coeficiente de Determinacion** ($R^2$) y el **Error Cuadratico Medio** (MSE).

---

## 2. Coeficiente de Determinacion ($R^2$)

El $R^2$ (o $r^2$) es la proporcion de la varianza en la variable dependiente ($y$) que es predecible a partir de las variables independientes ($X$).

### Definicion

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

donde:

- $SS_{res} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$ es la **suma de cuadrados de los residuos** (error del modelo)
- $SS_{tot} = \sum_{i=1}^{n}(y_i - \bar{y})^2$ es la **suma total de cuadrados** (varianza total de los datos)

### Interpretacion

| Valor de $R^2$ | Interpretacion |
|---|---|
| $R^2 = 1$ | El modelo explica el 100% de la varianza (ajuste perfecto) |
| $R^2 = 0.8$ | El modelo explica el 80% de la varianza |
| $R^2 = 0$ | El modelo no es mejor que predecir siempre la media |
| $R^2 < 0$ | El modelo es peor que predecir la media (posible en datos de prueba) |

En el caso de Regresion Lineal Simple, $R^2$ es simplemente el cuadrado del coeficiente de correlacion: $R^2 = (\rho_{X,Y})^2$.

In [ ]:
# Ejemplo visual: que significa R^2
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# R^2 alto
x1 = np.linspace(0, 10, 50)
y1 = 2 * x1 + 1 + rng.randn(50) * 1
model1 = LinearRegression().fit(x1.reshape(-1,1), y1)
r2_1 = model1.score(x1.reshape(-1,1), y1)
axes[0].scatter(x1, y1, alpha=0.6, color='steelblue')
axes[0].plot(x1, model1.predict(x1.reshape(-1,1)), 'r-', linewidth=2)
axes[0].set_title(f'R^2 = {r2_1:.3f} (Alto)', fontsize=12)

# R^2 medio
y2 = 2 * x1 + 1 + rng.randn(50) * 5
model2 = LinearRegression().fit(x1.reshape(-1,1), y2)
r2_2 = model2.score(x1.reshape(-1,1), y2)
axes[1].scatter(x1, y2, alpha=0.6, color='orange')
axes[1].plot(x1, model2.predict(x1.reshape(-1,1)), 'r-', linewidth=2)
axes[1].set_title(f'R^2 = {r2_2:.3f} (Medio)', fontsize=12)

# R^2 bajo
y3 = rng.randn(50) * 5
model3 = LinearRegression().fit(x1.reshape(-1,1), y3)
r2_3 = model3.score(x1.reshape(-1,1), y3)
axes[2].scatter(x1, y3, alpha=0.6, color='red')
axes[2].plot(x1, model3.predict(x1.reshape(-1,1)), 'r-', linewidth=2)
axes[2].set_title(f'R^2 = {r2_3:.3f} (Bajo)', fontsize=12)

for ax in axes:
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)

plt.suptitle('Visualizacion del Coeficiente de Determinacion R^2', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

### Calculo manual de $R^2$

In [ ]:
# Calculo manual vs sklearn
y_pred_1 = model1.predict(x1.reshape(-1,1))

# Manual
ss_res = np.sum((y1 - y_pred_1) ** 2)
ss_tot = np.sum((y1 - np.mean(y1)) ** 2)
r2_manual = 1 - (ss_res / ss_tot)

# Con sklearn
from sklearn.metrics import r2_score
r2_sklearn = r2_score(y1, y_pred_1)

print(f'R^2 calculado manualmente: {r2_manual:.6f}')
print(f'R^2 con sklearn:           {r2_sklearn:.6f}')
print(f'R^2 con model.score():     {r2_1:.6f}')

---

## 3. Error Cuadratico Medio (MSE) y RMSE

### MSE (Mean Squared Error)

El MSE calcula el promedio de los cuadrados de los errores (residuos):

$$MSE = \frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

### RMSE (Root Mean Squared Error)

El RMSE es la raiz cuadrada del MSE, lo cual lo devuelve a las **mismas unidades** que la variable objetivo:

$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

### Diferencias clave entre $R^2$ y MSE/RMSE

| Caracteristica | $R^2$ | MSE / RMSE |
|---|---|---|
| Rango | $(-\infty, 1]$ | $[0, +\infty)$ |
| Mejor valor | 1 | 0 |
| Unidades | Adimensional | Cuadradas / Originales |
| Interpretacion | Proporcion de varianza explicada | Error absoluto promedio |
| Comparabilidad | Entre modelos sobre los mismos datos | Solo dentro del mismo dataset |

In [ ]:
from sklearn.metrics import mean_squared_error

# Calcular MSE y RMSE para los tres modelos del ejemplo anterior
models = [(model1, y1, 'Alto'), (model2, y2, 'Medio'), (model3, y3, 'Bajo')]

print(f'{"Modelo":<10} {"R^2":>8} {"MSE":>10} {"RMSE":>10}')
print('-' * 40)

for model, y_true, name in models:
    y_pred = model.predict(x1.reshape(-1, 1))
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f'{name:<10} {r2:>8.4f} {mse:>10.4f} {rmse:>10.4f}')

---

## 4. Otras metricas complementarias

Ademas de $R^2$ y MSE, existen otras metricas utiles:

| Metrica | Formula | Descripcion |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Error absoluto medio. Menos sensible a outliers que MSE |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Error porcentual. Util para comparar entre datasets |
| **R^2 ajustado** | $1 - \frac{(1-R^2)(n-1)}{n-p-1}$ | Penaliza por el numero de variables. Util en RLM |

In [ ]:
from sklearn.metrics import mean_absolute_error

# Ejemplo con el modelo 1 (R^2 alto)
y_pred = model1.predict(x1.reshape(-1, 1))

r2 = r2_score(y1, y_pred)
mse = mean_squared_error(y1, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y1, y_pred)

print('Metricas de evaluacion:')
print(f'  R^2:  {r2:.4f}')
print(f'  MSE:  {mse:.4f}')
print(f'  RMSE: {rmse:.4f}')
print(f'  MAE:  {mae:.4f}')

---

## 5. Division entrenamiento/prueba y validacion

Un aspecto fundamental de la evaluacion es **no evaluar con los mismos datos que se usaron para entrenar**. Si lo hacemos, el modelo podria parecer perfecto simplemente porque memorizo los datos.

La practica estandar es dividir los datos en:

- **Conjunto de entrenamiento** (e.g., 70%): para ajustar el modelo.
- **Conjunto de prueba** (e.g., 30%): para evaluar el modelo con datos que nunca vio.

La diferencia entre las metricas en entrenamiento y prueba nos indica si hay sobreajuste:

- Si $R^2_{train} \gg R^2_{test}$: hay **sobreajuste** (el modelo memorizo).
- Si $R^2_{train} \approx R^2_{test}$: el modelo **generaliza bien**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Generar datos
rng = np.random.RandomState(1)
x = 10 * rng.rand(80)
y = np.sin(x) + 0.1 * rng.randn(80)

X = x[:, np.newaxis]

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Comparar modelos con diferentes grados
degrees = range(1, 16)
r2_train_list = []
r2_test_list = []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train, y_train)
    
    r2_train_list.append(model.score(X_train, y_train))
    r2_test_list.append(model.score(X_test, y_test))

# Visualizar
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(degrees), r2_train_list, 'o-', color='steelblue', label='R^2 Train', linewidth=2)
ax.plot(list(degrees), r2_test_list, 's-', color='red', label='R^2 Test', linewidth=2)
ax.set_xlabel('Grado del polinomio', fontsize=12)
ax.set_ylabel('R^2', fontsize=12)
ax.set_title('R^2 en Train vs Test segun grado polinomial', fontsize=13)
ax.legend(fontsize=11)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.set_ylim(-0.5, 1.1)
plt.tight_layout()
plt.show()

print('Observa como a partir de cierto grado, R^2_test comienza a degradarse')
print('mientras R^2_train sigue subiendo --> esto es sobreajuste.')

---

## 6. Ejemplo completo con el dataset Iris

Construiremos un modelo de regresion completo con el dataset Iris, evaluando con todas las metricas aprendidas. El objetivo sera predecir `petal_width` a partir de las demas variables.

In [ ]:
# Cargar datos
iris = sns.load_dataset('iris')

# Preparar X e y
features = ['sepal_length', 'sepal_width', 'petal_length']
target = 'petal_width'

X = iris[features].values
y = iris[target].values

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba: {X_test.shape[0]} muestras')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Crear pipeline: Estandarizar + Polinomio grado 2 + RL
model = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(2),
    LinearRegression()
)

# Entrenar
model.fit(X_train, y_train)

# Predecir
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
# Calcular todas las metricas
metrics = {
    'Conjunto': ['Entrenamiento', 'Prueba'],
    'R^2': [
        r2_score(y_train, y_pred_train),
        r2_score(y_test, y_pred_test)
    ],
    'MSE': [
        mean_squared_error(y_train, y_pred_train),
        mean_squared_error(y_test, y_pred_test)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_train, y_pred_train)),
        np.sqrt(mean_squared_error(y_test, y_pred_test))
    ],
    'MAE': [
        mean_absolute_error(y_train, y_pred_train),
        mean_absolute_error(y_test, y_pred_test)
    ]
}

df_metrics = pd.DataFrame(metrics)
print('Metricas de evaluacion del modelo:')
print(df_metrics.to_string(index=False))

In [ ]:
# Visualizacion final
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predicciones vs Reales
axes[0].scatter(y_test, y_pred_test, color='steelblue', alpha=0.7, edgecolors='black', linewidth=0.5)
axes[0].plot([0, 2.5], [0, 2.5], 'r--', linewidth=1.5)
axes[0].set_xlabel('Valor Real', fontsize=11)
axes[0].set_ylabel('Valor Predicho', fontsize=11)
axes[0].set_title(f'Predicciones vs Reales\nR^2 = {r2_score(y_test, y_pred_test):.3f}', fontsize=12)

# 2. Distribucion de residuos
residuals = y_test - y_pred_test
axes[1].hist(residuals, bins=12, edgecolor='black', color='steelblue', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residuo', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title('Distribucion de Residuos', fontsize=12)

# 3. Residuos vs Predichos
axes[2].scatter(y_pred_test, residuals, color='steelblue', alpha=0.7)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[2].set_xlabel('Valor Predicho', fontsize=11)
axes[2].set_ylabel('Residuo', fontsize=11)
axes[2].set_title('Residuos vs Predichos', fontsize=12)

plt.suptitle('Evaluacion del Modelo de Regresion - Dataset Iris', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

---

## Resumen

En este notebook aprendimos:

1. El **$R^2$** mide la proporcion de varianza explicada por el modelo (1 = perfecto, 0 = igual a predecir la media).
2. El **MSE** y **RMSE** miden el error promedio de las predicciones en unidades cuadradas y originales respectivamente.
3. El **MAE** es una alternativa menos sensible a outliers.
4. La **division train/test** es esencial para evaluar la capacidad de generalizacion del modelo.
5. La comparacion de metricas entre train y test permite detectar **sobreajuste**.
6. Un diagnostico completo incluye graficas de predicciones vs reales y analisis de residuos.

Con esto concluimos la serie de notebooks sobre Regresion Lineal. El siguiente paso es practicar con los ejercicios propuestos.